# Chg2Cap: Colab training and evaluation

Select a GPU runtime, edit the configuration cell, then run all cells in order.

Each run gets its own folder in Google Drive with `run.json` (code commit, configuration, dataset hash, seed, environment and results), logs, checkpoints and test results. See the README for details.

## 1. Configuration

In [ ]:
# Code version: branch, tag or commit. Use a tag or commit for results you will report.
REPO_URL = "https://github.com/yakupgulcan/Chg2Cap.git"
REPO_REF = "thesis"
OUTPUT_ROOT = "/content/drive/MyDrive/Chg2Cap_Outputs"  # must be under MyDrive

# Data
EXPERIMENT = "baseline"  # short name; becomes part of the run folder name
DATASET_SOURCE = "https://drive.google.com/file/d/1YgWES-0OOL-3KK4yIV3-y8MN-_c1ELEE/view?usp=drive_link"  # ZIP (URL or Drive path) with LevirCCcaptions.json and images/

# Training
SEED = 123  # None: choose a random seed and record it
TOTAL_EPOCHS = 20  # total target, including completed epochs when resuming
BATCH_SIZE = 32
NUM_WORKERS = 2
STOP_CRITERIA = 5  # epochs without validation BLEU-4 improvement before stopping; 0 disables
ENCODER_LR = 1e-4
DECODER_LR = 1e-4
MODEL_ARGS = []  # model arguments for train.py and test.py, e.g. ["--n_layers", "2"]
EXTRA_TRAIN_ARGS = []  # further train.py arguments
RESUME_CHECKPOINT = None  # Drive path of a last.pth to continue from
RUN_SMOKE_CHECK = False  # short train/resume/test check on four image pairs before training

# Evaluation
EVAL_CHECKPOINT = None  # None: best.pth of this run; otherwise a checkpoint path
DISCONNECT_WHEN_DONE = True  # release the runtime after the last cell

# Saved in run.json. Only plain settings are kept, so re-running this cell later stays harmless.
CONFIG = {name: value for name, value in dict(globals()).items()
          if name.isupper() and isinstance(value, (str, int, float, bool, list, type(None)))}

## 2. Setup

Mounts Drive, clones the code and installs dependencies. Use a fresh runtime for each run.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import auth, drive

drive.mount("/content/drive")
auth.authenticate_user()  # Drive API access for checkpoint uploads

PROJECT_DIR = Path("/content/Chg2Cap")
if PROJECT_DIR.exists():
    raise FileExistsError(f"{PROJECT_DIR} already exists. Use a fresh runtime.")
subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=PROJECT_DIR, check=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
%pip install --quiet -r requirements-colab.txt gdown google-api-python-client google-auth

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab.")
if shutil.which("java") is None:  # METEOR needs Java
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jre-headless", shell=True, check=True)
subprocess.run([sys.executable, "scripts/prepare_meteor.py"], check=True)

## 3. Start the run

Creates the run folder and `run.json`.

In [ ]:
import json, secrets
from datetime import datetime, timezone
from drive_checkpoints import DriveCheckpoints, checksum, drive_service, resolve_folder
from runtime import (extract_raw_dataset, fetch_archive, run_logged, start_run,
                     training_summary, update_run)


def timestamp():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


RUN_SEED = SEED if SEED is not None else secrets.randbits(32)
RUN_ID = f"{EXPERIMENT}_seed{RUN_SEED}_{timestamp()}"
RUN_DIR = Path(OUTPUT_ROOT) / "runs" / RUN_ID  # Drive: records, logs, checkpoints, results
WORK_DIR = PROJECT_DIR / "runs" / RUN_ID  # runtime storage: dataset and local checkpoints
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
os.environ["PYTHONHASHSEED"] = str(RUN_SEED)

start_run(RUN_DIR, {**CONFIG, "SEED": RUN_SEED}, PROJECT_DIR)
CHECKPOINT_DIR.mkdir(parents=True)
(RUN_DIR / "checkpoints").mkdir()
service = drive_service()
# The folder was just created through the mount; wait until Drive itself lists it.
DRIVE_CHECKPOINT_FOLDER_ID = resolve_folder(service, RUN_DIR / "checkpoints", wait_seconds=300)
print((RUN_DIR / "run.json").read_text())

## 4. Prepare the data

Downloads the dataset and writes caption tokens, split lists and the vocabulary.

In [ ]:
archive, archive_sha256 = fetch_archive(DATASET_SOURCE, "/content/dataset.zip", RUN_DIR / "data.log")
dataset_dir = extract_raw_dataset(archive, WORK_DIR / "raw_dataset")
archive.unlink()
run_logged([sys.executable, "-u", "preprocess_data.py", "--dataset", "LEVIR_CC",
            "--dataset_root", str(dataset_dir), "--word_count_threshold", "5"], RUN_DIR / "data.log")

manifest = json.loads((PROJECT_DIR / "data/LEVIR_CC/manifest.json").read_text())
if manifest["max_length"] > 41:
    raise ValueError("Captions exceed max_length=41; pass --max_length consistently to train and test.")
update_run(RUN_DIR, data={"source": DATASET_SOURCE, "archive_sha256": archive_sha256,
                          "preprocessing": manifest})
print("Dataset archive SHA-256:", archive_sha256)
DATA_ARGS = ["--data_folder", str(dataset_dir / "images"), "--workers", str(NUM_WORKERS), *MODEL_ARGS]

if RUN_SMOKE_CHECK:
    run_logged([sys.executable, "-u", "scripts/smoke_check.py", "--dataset_root", str(dataset_dir),
                "--output_dir", str(WORK_DIR / "smoke"), "--resume_check"], RUN_DIR / "smoke.log")

## 5. Train

Validation runs after every epoch. `last.pth` and `best.pth` are uploaded to the run folder.

In [ ]:
command = [sys.executable, "-u", "train.py", *DATA_ARGS,
           "--savepath", str(CHECKPOINT_DIR), "--drive_checkpoint_folder_id", DRIVE_CHECKPOINT_FOLDER_ID,
           "--num_epochs", str(TOTAL_EPOCHS), "--patience", str(STOP_CRITERIA),
           "--train_batchsize", str(BATCH_SIZE), "--encoder_lr", str(ENCODER_LR),
           "--decoder_lr", str(DECODER_LR), "--seed", str(RUN_SEED), *EXTRA_TRAIN_ARGS]
if RESUME_CHECKPOINT:
    remote = Path(RESUME_CHECKPOINT)
    local = WORK_DIR / "resume_source" / remote.name
    DriveCheckpoints(resolve_folder(service, remote.parent), service).download(remote.name, local)
    command += ["--checkpoint", str(local)]

try:
    run_logged(command, RUN_DIR / "train.log")
finally:
    records = list(CHECKPOINT_DIR.glob("training_epoch_*.json"))
    if records:
        (RUN_DIR / "epochs").mkdir(exist_ok=True)
        for record in records:
            shutil.copy2(record, RUN_DIR / "epochs")
        update_run(RUN_DIR, training={"command": command, **training_summary(CHECKPOINT_DIR)})

## 6. Evaluate on the test set

In [ ]:
checkpoint = Path(EVAL_CHECKPOINT) if EVAL_CHECKPOINT else CHECKPOINT_DIR / "best.pth"
if not checkpoint.is_file():
    raise FileNotFoundError(checkpoint)
results_dir = RUN_DIR / "eval_results" / timestamp()
run_logged([sys.executable, "-u", "test.py", *DATA_ARGS, "--savepath", str(checkpoint.parent),
            "--checkpoint", checkpoint.name, "--results_dir", str(results_dir)], RUN_DIR / "eval.log")

test_metrics = json.loads((results_dir / "metrics.json").read_text())
update_run(RUN_DIR, test={"checkpoint": str(checkpoint), "checkpoint_md5": checksum(checkpoint),
                          "results": results_dir.name, "metrics": test_metrics})
print(json.dumps(test_metrics, indent=2))

## 7. Finish

Marks the run completed and, with `DISCONNECT_WHEN_DONE=True`, releases the runtime. Runtime-local files are lost; everything needed is in the run folder.

In [ ]:
update_run(RUN_DIR, status="completed", finished_utc=datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Run folder:", RUN_DIR)
if DISCONNECT_WHEN_DONE:
    from google.colab import runtime
    drive.flush_and_unmount()
    runtime.unassign()